#   Chatbot using LangGraph
`we use a concept in this code called `persistence` from langgraph whch basicaly hold the previous context`
`To remove error remove the checkpoint code concept`

In [87]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
import os
import operator
from langgraph.checkpoint.memory import MemorySaver #persistence m/m


In [88]:
llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4o-mini")

In [89]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [90]:
def chat_node(state: ChatState):
    #take user query from state
    messages = state['messages']

    #send to llm
    response = llm.invoke(messages)

    #response store to state
    return {'messages': [response]}

In [91]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)
# chatbot



In [92]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
thread_id = 1

while True:
    user_message = input('Type Here: ')
    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    
    config = {
        'configurable': {'thread_id': thread_id}
    }
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI:', response['messages'][-1].content)

User: hi my name is deepak
AI: Hello Deepak! How can I assist you today?
User: what is my name
AI: Your name is Deepak. How can I help you today, Deepak?
User: can you add n to 100
AI: Sure! If you add \( n \) to 100, the expression would be \( 100 + n \). If you have a specific value for \( n \) that you want to add, please let me know!
User: can you multiply 100 with 2
AI: Yes! Multiplying 100 by 2 gives you:

\[ 100 \times 2 = 200 \]

If you have any more questions or need further calculations, feel free to ask!
User: bye


In [ ]:
# view all message
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is deepak', additional_kwargs={}, response_metadata={}, id='df5bd332-92f5-4dc9-a4eb-5d46bd31290f'), AIMessage(content='Hello Deepak! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e3916b0f3e', 'id': 'chatcmpl-DhZLYoGyyMzwRWT9SE27ZacscCbix', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4522-7a41-74d0-a804-3cfa50829c87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 11, 'total_tokens': 24, 'input_token_details': {'audio'